In [1]:
import pandas as pd
import numpy as np 
import seaborn as sns
import matplotlib.pyplot as plt 
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os

# Cargar variables del .env
load_dotenv('pg_credenciales.env')

engine = create_engine(
    f"postgresql+psycopg2://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}@"
    f"{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
)

ValueError: invalid literal for int() with base 10: 'None'

In [ ]:
consulta="""
SELECT table_name
FROM information_schema.tables
WHERE table_schema='gob_ciencia_y_tecnologia'

"""

pd.read_sql_query(consulta,con=engine)

tablas = pd.read_sql_query(consulta,con=engine)

In [ ]:
display(tablas)

In [ ]:
# Celda 2: Ver la estructura de cada tabla
for tabla in tablas['table_name']:
    print(f"\n{'='*60}")
    print(f"📋 ESTRUCTURA DE: {tabla}")
    print('='*60)
    
    # Obtener columnas y tipos de datos
    consulta_estructura = f"""
    SELECT 
        column_name,
        data_type,
        character_maximum_length,
        is_nullable
    FROM information_schema.columns 
    WHERE table_schema = 'gob_ciencia_y_tecnologia' 
      AND table_name = '{tabla}'
    ORDER BY ordinal_position;
    """
    
    estructura = pd.read_sql_query(consulta_estructura, con=engine)
    display(estructura)
    
    # Ver primeras 3 filas como ejemplo
    consulta_muestra = f"""
    SELECT * 
    FROM gob_ciencia_y_tecnologia."{tabla}" 
    LIMIT 3;
    """
    
    muestra = pd.read_sql_query(consulta_muestra, con=engine)
    print(f"\nPrimeras filas de {tabla}:")
    display(muestra)

In [ ]:
#ANALISIS EXPLORATORIO 

#NUMERO DE REGISTROS POR TABLA
print("resumen de registros por tabla")
for tabla in tablas['table_name']:
    consulta_conteo=f'SELECT COUNT(*) as total FROM gob_ciencia_y_tecnologia."{tabla}" '
    resultado=pd.read_sql_query(consulta_conteo, con=engine)
    print(f"-{tabla}:{resultado['total'][0]:,} registros")

In [ ]:
query_muestra=f' SELECT * FROM gob_ciencia_y_tecnologia."{tabla}" LIMIT 3'
muestra =pd.read_sql_query(query_muestra, con = engine)
print("Muestra de datos:")
display(muestra)

In [ ]:
nombres_tablas=tablas['table_name'].tolist()
tablas_snii = [t for t in nombres_tablas if 'snii' in t.lower()]
print(f'tablas con snii: {tablas_snii}')


for tabla in tablas_snii:
    query = f"""
       SELECT area_conocimiento, 
            COUNT(*) AS cantidad,
            ROUND(100.0 * COUNT(*)/SUM(COUNT(*)) OVER(),2) AS porcentaje   
        FROM gob_ciencia_y_tecnologia.{tabla}
        
        GROUP BY area_conocimiento
        ORDER BY area_conocimiento
        """

    temp=pd.read_sql_query(query, con = engine)
    print(temp)

    labels_short = [label[:30] + '...' if len(label) > 20 else label for label in temp['area_conocimiento']]
    plt.figure(figsize=(20,10))

    temp_short = temp.copy()
    temp_short['area_conocimiento_short'] = labels_short

    plt.figure(figsize=(20,10))

    plt.subplot(1,2,1)
    sns.barplot(x='area_conocimiento_short', y='cantidad', data=temp_short)
    plt.xticks(rotation=45, ha='right')
    plt.xlabel('Área de conocimiento')
    plt.title("Distribucion de areas de conocimiento")

    plt.subplot(1,2,2)
    plt.pie(temp['porcentaje'], labels=labels_short, autopct='%1.1f%%')
    plt.title('porcentaje por area de conocimiento')

    plt.tight_layout()
    plt.show()


In [ ]:
tablas_publicaciones = [t for t in nombres_tablas if 'publicaciones' in t.lower()]
print(f'tablas de publicaciones :{tablas_publicaciones}')

for tabla in tablas_publicaciones:
    query= f"""
    SELECT  column_name 
    FROM information_schema.columns 
    WHERE table_schema = 'gob_ciencia_y_tecnologia' 
    AND table_name='{tabla}' 
    """
    resultado=pd.read_sql_query(query,con=engine)
    
    display(resultado)
   
    
    if not resultado.empty:
        print ('---------------------')

In [ ]:
# Crear una lista vacía para almacenar los DataFrames
dfs = []

for tabla in tablas_publicaciones:
    for col_tipo in ['tipo_publicacion', 'tipo']:
        try:
            query = f"""
                SELECT
                    anio,
                    {col_tipo} as tipo_publicacion,
                    COUNT(*) as cantidad,
                    '{tabla}' as tabla_origen
                FROM gob_ciencia_y_tecnologia."{tabla}"
                WHERE {col_tipo} IS NOT NULL
                GROUP BY anio, {col_tipo}
                ORDER BY anio, cantidad DESC
            """
            df_tipos = pd.read_sql_query(query, con=engine)
            dfs.append(df_tipos)  # Agregar a la lista
            #print(f"✅ {tabla} - agregada usando columna: {col_tipo}")
            break  # Salir del bucle si funcionó
        except Exception as e:
          #  print(f"⚠️ {tabla} - falló con {col_tipo}: {e}")
            continue  # Probar la siguiente columna

# Concatenar todos los DataFrames
df_todos_los_tipos = pd.concat(dfs, ignore_index=True)


df_todos_los_tipos

In [ ]:
import warnings

warnings.filterwarnings('ignore')

# Calcular total por tipo
total_por_tipo = df_todos_los_tipos.groupby('tipo_publicacion')['cantidad'].sum().reset_index()
total_por_tipo = total_por_tipo.sort_values('cantidad', ascending=False)

plt.figure(figsize=(12, 6))
sns.barplot(data=total_por_tipo, x='tipo_publicacion', y='cantidad', palette='viridis')
plt.title('Total de publicaciones por tipo', fontsize=14)
plt.xlabel('Tipo de publicación')
plt.ylabel('Cantidad total')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
for tabla in tablas_snii:
    query= f"""
    SELECT  column_name 
    FROM information_schema.columns 
    WHERE table_schema = 'gob_ciencia_y_tecnologia' 
    AND table_name='{tabla}' 
    """
    resultado=pd.read_sql_query(query,con=engine)
    display(resultado)
    if not resultado.empty:
        print('-------------')

In [ ]:
#tablas_snii  'snii_s1_2025'

In [ ]:
# Celda 6: Query de distribución por nivel SNI
# REEMPLAZA 'investigadores_sni_2025_1' y 'investigadores_sni_2025_2' 
# con los nombres reales de tus tablas 

query_nivel_sni = """
-- Unimos ambos semestres y agregamos etiqueta de semestre
WITH investigadores_total AS (
    SELECT *, 'Semestre 1' as semestre 
    FROM gob_ciencia_y_tecnologia.snii_s1_2025
    UNION ALL
    SELECT *, 'Semestre 2' as semestre 
    FROM gob_ciencia_y_tecnologia.snii_s2_2025
)
SELECT 
    semestre,
    nivel,
    COUNT(*) as total_investigadores,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (PARTITION BY semestre), 2) as porcentaje
FROM investigadores_total
WHERE nivel IS NOT NULL
GROUP BY semestre, nivel
ORDER BY semestre, 
    CASE nivel
        WHEN 'C' THEN 1
        WHEN '1' THEN 2
        WHEN '2' THEN 3
        WHEN '3' THEN 4
        WHEN 'E' THEN 5
        ELSE 6
    END;
"""

# Ejecutar query
df_nivel = pd.read_sql_query(query_nivel_sni, con=engine)

# Visualización
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Gráfico de barras comparativo
sns.barplot(data=df_nivel, x='nivel', y='total_investigadores', 
            hue='semestre', palette='viridis', ax=ax1)
ax1.set_title('Distribución de Investigadores SNI por Nivel (2025)', fontsize=14, fontweight='bold')
ax1.set_xlabel('Nivel SNI')
ax1.set_ylabel('Cantidad de Investigadores')
ax1.legend(title='Período')

#Gráfico de pastel para el semestre más reciente
df_s2 = df_nivel[df_nivel['semestre'] == 'Semestre 2']
ax2.pie(df_s2['total_investigadores'], labels=df_s2['nivel'], autopct='%1.1f%%', 
        startangle=90, colors=sns.color_palette('viridis', len(df_s2)))
ax2.set_title('Proporción por Nivel - Semestre 2, 2025', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()



In [ ]:
df_nivel = pd.read_sql_query(query_nivel_sni, con=engine)
df_nivel.info()

In [ ]:
df_nivel.head()

In [ ]:
print("Distribución por Nivel SNI:")
display(df_nivel.pivot_table(values='total_investigadores', 
                             index='nivel', 
                             columns='semestre', 
                             aggfunc='sum'))

In [ ]:
plt.figure(figsize=(14, 6))
sns.lineplot(data=df_todos_los_tipos, x='anio', y='cantidad', hue='tipo_publicacion', marker='o')
plt.title('Evolución de publicaciones por tipo', fontsize=14)
plt.xlabel('Año')
plt.ylabel('Cantidad')
plt.legend(title='Tipo de publicación', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, alpha=0.3)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()